# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas
**Material:** *Preparar y estructurar datos con Python*, lecciones 2, 3 y 4
**Herramientas:** Python · Pandas · PyArrow · Jupyter Notebook

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno continúa el proyecto de la actividad anterior ([`data_quality.ipynb`](data_quality.ipynb)), donde se exploraron los cinco archivos sin modificarlos. Aquí se resuelven los problemas encontrados y se preparan los datos para el análisis.

**Contenido**

0. Preparación del entorno y carga de los datos
1. Lección 2 · Limpieza y estandarización
   - Ejemplos de la lección
   - Laboratorio guiado 2: pipeline de limpieza con informe de cambios
   - Reto 2
   - Documentación de la limpieza
2. Lección 3 · Filtrado, ordenamiento y agrupación
   - Ejemplos de la lección
   - Laboratorio guiado 3: perfil de hábitos de compra
   - Reto 3
   - Interpretación de los resultados
3. Lección 4 · Combinación de datasets
   - Ejemplos de la lección
   - Laboratorio guiado 4: tabla de análisis por producto
   - Reto 4
   - Validación y archivos procesados
4. Conclusiones

> **Cómo ejecutar:** copie los cinco CSV en `data/raw/`, abra este cuaderno desde `notebooks/` y use *Kernel → Restart & Run All*. Las lecciones se ejecutan en orden porque cada una usa las variables de la anterior. Los archivos generados se guardan en `data/processed/`.

---
# 0. Preparación del entorno y carga de los datos

Las rutas son relativas a `notebooks/`: los datos originales se leen de `data/raw/` y los resultados se guardan en `data/processed/`. Los archivos usan punto y coma (`;`) como separador.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)

print('pandas', pd.__version__)

pandas 3.1.0


In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Las cinco tablas tienen los tamaños originales, los mismos de la actividad anterior. El diagnóstico de calidad está en [`reports/data_quality.md`](../reports/data_quality.md): 15 pedidos duplicados, nulos en tres columnas, columnas enteras leídas como `float64` y el departamento centinela `missing`. Esa es la lista de tareas de la Lección 2.

---
# 1. Lección 2 · Limpieza y estandarización

Para cada problema hay tres opciones: **eliminar, corregir o conservar**. La decisión depende de qué significa el dato. El orden de trabajo es: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

## 1.1 Ejemplos de la lección

### Duplicados exactos: `duplicated` y `drop_duplicates`

Primero se detectan los duplicados, sin borrar nada, para ver qué son.

In [3]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


**Resultado:** 30 filas, es decir, 15 pedidos y sus 15 copias. Todas tienen `order_dow = 3` y `order_hour_of_day = 2` (miércoles a las 2 a. m.). Que todas ocurran en la misma franja apunta a un fallo del sistema y no a pedidos reales, así que es seguro **eliminarlas**.

In [4]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### Tipos de dato con `astype`

Las columnas enteras se convierten al tipo más pequeño que admite sus valores. Las que tienen nulos usan `Int16` (con mayúscula), el tipo entero *nullable*.

In [5]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


**Resultado:** la memoria de `orders` baja de 23,0 MB a 7,2 MB sin perder información. `int8` alcanza para días (0–6) y horas (0–23), `int16` para `order_number` (máximo 100) e `int32` para los identificadores.

### Nulos: comprobar la causa antes de decidir

Hipótesis: `days_since_prior_order` es nulo solo en el primer pedido de cada cliente.

In [6]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


In [7]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


**Resultado:** la hipótesis se cumple (`True`): cada nulo de `days_since_prior_order` es un primer pedido, así que se **conserva**; rellenarlo con 0 inventaría que el cliente volvió el mismo día y bajaría la espera media. Los 1.258 productos sin nombre están todos en el pasillo 100 y el departamento 21 (`missing`); se **corrigen** con `'Unknown'` porque el producto existe y no se puede borrar.

| Decisión | Columna | Causa |
|---|---|---|
| Conservar el nulo | `days_since_prior_order` | Primer pedido del cliente: no existe pedido anterior |
| Rellenar con `'Unknown'` | `product_name` | Producto sin datos de catálogo (pasillo y departamento `missing`) |
| Conservar el nulo | `add_to_cart_order` | Se investiga en el Reto 2, punto 3 |

### Texto con los métodos `.str`

In [8]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


**Resultado:** había 403 nombres con espacios dobles y 16 con el espacio invisible `\xa0`; después de limpiar quedan 0 y 0. El reemplazo de `\xa0` va en un paso propio porque en pandas 3 la expresión `\s` no lo reconoce.

### Duplicados implícitos

Algunos nombres solo se repiten después de estandarizar, como `American Cheese slices` y `American Cheese Slices`.

In [9]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


**Resultado:** 141 nombres repetidos, y 268 de los 280 productos implicados aparecen en pedidos. La decisión es **conservar y documentar**: tienen `product_id` distintos y borrarlos dejaría pedidos apuntando a productos inexistentes. En los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [10]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


Aplicada a una tabla que ya está limpia, la función no cambia nada: la limpieza es repetible. Verificación de toda la limpieza (si la celda no muestra nada, todo está correcto):

In [11]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

## 1.2 Laboratorio guiado 2: pipeline de limpieza con informe de cambios

**Encargo:** construir un *pipeline*, una secuencia de pasos de limpieza encadenados que parte del archivo crudo, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso. Se trabaja sobre copias nuevas de los archivos, sin tocar las tablas ya limpias.

**Paso 1. Recargar los datos crudos**

In [12]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6)

 (49694, 4) (4545007, 4)


**Paso 2. Una función que anota el estado de la tabla**

`anotar` guarda una "foto" de la tabla (filas, nulos y memoria) y la devuelve sin cambios, para poder intercalarla entre pasos.

In [13]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

**Paso 3. Un paso de limpieza por función**

Cada función recibe un DataFrame y devuelve otro; esa regla es la que permite encadenarlas.

In [14]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

**Paso 4. Encadenar los pasos con `pipe`**

In [15]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

**Paso 5. El informe de cambios**

In [16]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- La limpieza de nombres eliminó los 1.258 nulos de `products` sin perder filas.
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos.

**Paso 6. Validar el resultado**

In [17]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


**Resultado:** `Validación superada` y `True True`. El pipeline produce exactamente las mismas tablas que la limpieza manual de los ejemplos, así que la limpieza se puede repetir con una sola instrucción cuando lleguen datos nuevos.

**Conclusiones clave del laboratorio 2**

- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- Intercalar una función de registro produce un informe de cambios sin esfuerzo extra.
- El informe responde con cifras a la pregunta "¿qué le hiciste a los datos?".
- Validar con `assert` al final convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## 1.3 Reto 2

### Punto 1. Duplicados y valores de `order_prods`

¿Tiene `order_prods` filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?

In [18]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))
print('¿Solo 0 y 1?', order_prods['reordered'].isin([0, 1]).all())

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿Solo 0 y 1? True


**Respuesta:** `order_prods` no tiene filas duplicadas exactas (0) ni pares `order_id` + `product_id` repetidos (0): cada producto aparece una sola vez por pedido. `reordered` solo contiene 0 y 1, así que no necesita corrección.

### Punto 2. Nombres de `aisles` y `departments`

¿Hay nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?

In [19]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    texto = df[col]
    print(nombre)
    print('  nombres duplicados:', texto.duplicated().sum())
    print('  duplicados sin distinguir mayúsculas:', texto.str.lower().str.strip().duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())

aisles
  nombres duplicados: 0
  duplicados sin distinguir mayúsculas: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
departments
  nombres duplicados: 0
  duplicados sin distinguir mayúsculas: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0


**Respuesta:** ni `aisles` (134 pasillos) ni `departments` (21 departamentos) tienen nombres duplicados, tampoco al ignorar mayúsculas, ni espacios al inicio, al final o dobles. **No necesitan limpieza de texto.** Lo único a tener en cuenta es el valor centinela `missing`, que es un dato válido como texto.

### Punto 3. Los 836 nulos de `add_to_cart_order`

¿En cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno de esos pedidos? ¿Qué los explica?

In [20]:
nulos_carrito = order_prods['add_to_cart_order'].isna()
pedidos_con_nulos = order_prods.loc[nulos_carrito, 'order_id'].unique()
tamano_pedido = order_prods.groupby('order_id').size()

print('Pedidos con nulos:', len(pedidos_con_nulos))
print('Productos del pedido más pequeño con nulos:', tamano_pedido.loc[pedidos_con_nulos].min())
print('Máximo valor registrado de add_to_cart_order:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos:', (tamano_pedido > 64).sum())

Pedidos con nulos: 70
Productos del pedido más pequeño con nulos: 65
Máximo valor registrado de add_to_cart_order: 64
Pedidos con más de 64 productos: 70


**Respuesta:** los 836 nulos están en **70 pedidos**, y el más pequeño de ellos tiene **65 productos**. El valor máximo registrado de `add_to_cart_order` es 64 y hay exactamente 70 pedidos con más de 64 productos. Esto indica que el sistema solo registra la posición en el carrito hasta el puesto 64: a partir del artículo 65 la posición queda vacía. No es un error del pedido, sino un límite del registro, por eso se conservan los nulos.

### Punto 4. Nombres que cambiaron con la estandarización

Se carga de nuevo `products.csv` y se comparan los nombres no nulos antes y después de limpiar.

In [21]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

con_nombre = products_crudo['product_name'].notna()
cambiaron = products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']
print('Nombres que cambiaron:', cambiaron.sum(), 'de', con_nombre.sum())

ejemplos = pd.DataFrame({'antes': products_crudo.loc[con_nombre, 'product_name'][cambiaron],
                         'después': products.loc[con_nombre, 'product_name'][cambiaron]})
ejemplos.head(5)

Nombres que cambiaron: 6321 de 48436


,antes,después
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Respuesta:** **6.321 de los 48.436 nombres** con valor (13,1 %) cambiaron con la estandarización. La mayoría de los cambios son de mayúsculas (`with` → `With`, `mcg` → `Mcg`), además de los espacios dobles e invisibles.

### Punto 5. Día de la semana como categoría ordenada

In [22]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
dia_texto = orders['order_dow'].map(dias)
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=list(dias.values()), ordered=True),
                          name='dia')

print(dia_categoria.dtype)
print(dia_categoria.cat.categories.tolist())
print('Memoria como texto:', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Memoria como categoría:', round(dia_categoria.memory_usage(deep=True) / 1e6, 2), 'MB')
print('¿domingo < sábado?', dia_categoria.min(), '<', dia_categoria.max())

category
['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']
Memoria como texto: 7.14 MB
Memoria como categoría: 0.48 MB
¿domingo < sábado? domingo < sábado


**Respuesta:** como texto, la columna del día ocupa 7,14 MB; como categoría ordenada, 0,48 MB, unas 15 veces menos, porque guarda cada nombre una sola vez y un código pequeño por fila. Además, al ser **ordenada**, pandas sabe que domingo va antes que sábado, así que `sort_values`, `min` y `max` respetan el orden de la semana y no el alfabético.

### Punto 6. Productos `'Unknown'` en los pedidos

In [23]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]

print('Filas de order_prods con productos Unknown:', len(filas_unknown))
print('% del total de filas:', round(len(filas_unknown) / len(order_prods) * 100, 2))
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())
print('% de los pedidos con productos:', round(filas_unknown['order_id'].nunique() / order_prods['order_id'].nunique() * 100, 2))

Filas de order_prods con productos Unknown: 9547
% del total de filas: 0.21
Pedidos en los que aparecen: 8254


% de los pedidos con productos: 1.83


**Respuesta:** hay **9.547 filas** de `order_prods` con productos `'Unknown'` (0,21 % del total), repartidas en **8.254 pedidos** (1,83 % de los pedidos con productos). **No los eliminaría:** son compras reales; borrarlas reduciría el tamaño de esos pedidos y alteraría la tasa de recompra y las posiciones en el carrito. Basta con identificarlos como `'Unknown'` y excluirlos solo en los análisis que dependan del nombre del producto.

### Punto 7. Reemplazar `'missing'` por un nulo real en una copia de `departments`

In [24]:
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print(departments_na.tail(3))
print('Nulos:', departments_na['department'].isna().sum())

# Efecto al agrupar: los productos del departamento nulo desaparecen del resultado
prod_depto = products.merge(departments_na, on='department_id')
print('Productos en total:', len(prod_depto))
print('Productos que quedan al agrupar:', prod_depto.groupby('department').size().sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Nulos: 1
Productos en total: 49694
Productos que quedan al agrupar: 48436


**Respuesta:** no conviene hacerlo en la tabla original porque `groupby` descarta los grupos nulos por defecto: al agrupar por departamento, los 1.258 productos de `missing` desaparecerían del resultado (quedan 48.436 de 49.694) y los totales dejarían de cuadrar.

## 1.4 Documentación de la limpieza

Resumen de las decisiones tomadas y su efecto. El informe del pipeline se guarda en `data/processed/cleaning_report.csv` y las tablas limpias en formato Parquet, que conserva los tipos de dato corregidos.

| Tabla | Problema | Decisión | Resultado |
|---|---|---|---|
| `orders` | 15 filas duplicadas (miércoles a las 2 a. m.) | Eliminar | 478.967 → 478.952 filas; `order_id` único |
| `orders` | Enteros leídos como `int64`/`float64` | Corregir tipos (`int8`/`int16`/`int32`, `Int16`) | 23,0 → 7,2 MB |
| `orders` | 28.817 nulos en `days_since_prior_order` | Conservar | Son primeros pedidos; rellenar con 0 falsearía la espera media |
| `products` | 1.258 nulos en `product_name` | Rellenar con `'Unknown'` | 0 nulos; los productos se conservan |
| `products` | Espacios dobles, espacios invisibles y mayúsculas irregulares | Estandarizar con `.str` | 6.321 nombres cambiados; 0 espacios dobles o invisibles |
| `products` | 141 nombres repetidos tras estandarizar | Conservar y documentar | Tienen `product_id` distintos; se agrupa por `product_id` |
| `order_prods` | Enteros en `int64`/`float64` | Corregir tipos | 145,4 → 54,5 MB |
| `order_prods` | 836 nulos en `add_to_cart_order` | Conservar | Posiciones no registradas en pedidos de más de 64 productos |
| `departments` | Valor centinela `missing` | Conservar como texto | Evita perder esos productos al agrupar |

In [25]:
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
orders.to_parquet(SALIDA / 'orders_clean.parquet', index=False)
products.to_parquet(SALIDA / 'products_clean.parquet', index=False)
order_prods.to_parquet(SALIDA / 'order_products_clean.parquet', index=False)

for archivo in sorted(SALIDA.glob('*')):
    if archivo.name != '.gitkeep':
        print(f'{archivo.name:32} {archivo.stat().st_size / 1e6:6.1f} MB')

cleaning_report.csv                 0.0 MB
order_products_clean.parquet       32.5 MB
orders_clean.parquet                5.4 MB
products_clean.parquet              1.5 MB
